In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import DataLoader, TensorDataset


In [6]:
import sys

sys.path.append('PatchTST_supervised')

In [2]:
# Загружаем данные
df = pd.read_csv('SBER_test.csv')
data = df[['open','close','high','low','value','volume']].values

# Масштабируем
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Формируем последовательности (пример: 128 прошлых баров для прогноза 16)
seq_len = 128
pred_len = 16

X, y = [], []
for i in range(len(data_scaled) - seq_len - pred_len):
    X.append(data_scaled[i:i+seq_len])
    y.append(data_scaled[i+seq_len:i+seq_len+pred_len, :4]) # прогноз open, close, high, low

X = np.array(X)
y = np.array(y)

# В DataLoader
dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [24]:
class Config:
    def __init__(self):
        self.enc_in = 4  # количество входных каналов
        self.seq_len = seq_len  # длина входной последовательности
        self.pred_len = pred_len  # длина предсказания
        self.e_layers = 3
        self.n_heads = 4
        self.d_model = 64
        self.d_ff = 256
        self.dropout = 0.1
        self.fc_dropout = 0.1
        self.head_dropout = 0.1
        self.individual = False
        self.patch_len = 24
        self.stride = 24
        self.padding_patch = True
        self.revin = True
        self.affine = False
        self.subtract_last = False
        self.decomposition = True
        self.kernel_size = 25

configs = Config()

In [27]:
from PatchTST_supervised.models.PatchTST import Model as PatchTST

model = PatchTST(
  configs=configs
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()


for epoch in range(50):
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        out = model(batch_X)
        # loss = loss_fn(out, batch_y)
        # Например, если предиктируем только последние значения
        loss = loss_fn(out[:, -batch_y.size(1):, :batch_y.size(2)], batch_y)

        loss.backward()
        optimizer.step()

In [30]:
# Загружаем данные
df = pd.read_csv('SBER_check.csv')
data = df[['open','close','high','low','value','volume']].values

# Масштабируем
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Формируем последовательности (пример: 128 прошлых баров для прогноза 16)
seq_len = 128
pred_len = 16

X, y = [], []
for i in range(len(data_scaled) - seq_len - pred_len):
    X.append(data_scaled[i:i+seq_len])
    y.append(data_scaled[i+seq_len:i+seq_len+pred_len, :4]) # прогноз open, close, high, low

X = np.array(X)
y = np.array(y)

# В DataLoader
dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))
test_loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [34]:
import matplotlib.pyplot as plt
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
with torch.no_grad():
    # Предположим, у вас есть test_loader
    batch_test_X, batch_test_y = next(iter(test_loader))
    batch_test_X, batch_test_y = batch_test_X.to(device), batch_test_y.to(device)
    
    pred = model(batch_test_X)  # может потребоваться привести размерности, как при обучении
    pred = pred[:, -batch_test_y.size(1):, :batch_test_y.size(2)]
    
    # возьмем первый пример из батча
    true_series = batch_test_y[0].cpu().numpy()
    true_series = scaler.inverse_transform(true_series)
    pred_series = pred[0].cpu().numpy()
    pred_series = scaler.inverse_transform(pred_series)
    
    time_steps = range(pred_series.shape[0])
    
    plt.figure(figsize=(12,6))
    for channel in range(true_series.shape[1]):
        plt.plot(time_steps, true_series[:, channel], label=f'True channel {channel}')
        plt.plot(time_steps, pred_series[:, channel], label=f'Pred channel {channel}', linestyle='--')
    plt.legend()
    plt.title("Сравнение предсказания и реальных данных")
    plt.xlabel("Временные шаги")
    plt.ylabel("Значение")
    plt.show()


ValueError: operands could not be broadcast together with shapes (16,4) (6,) (16,4) 